In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
let divide (numerator: float) (denominator: float) =
    if denominator = 0.0 then failwith "Division by zero"
    numerator / denominator

# A Simple Calculator

This file shows the implementation of a *symbolic calculator* using `FsLexYacc`.  The grammar for the language implemented by this parser is as follows:
$$
\begin{array}{lcl}
  \texttt{stmnt}   & \rightarrow & \;\texttt{IDENTIFIER}\; \texttt{':='} \; \;\texttt{expr}\; \texttt{';'} \\
                   & \mid        & \;\texttt{expr}\; \texttt{';'} \\[0.2cm]   
  \texttt{expr}    & \rightarrow & \;\texttt{expr}\; \texttt{'+'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{expr}\; \texttt{'-'} \; \texttt{product}  \\
                   & \mid        & \;\texttt{product}                                  \\[0.2cm]
  \texttt{product} & \rightarrow & \;\texttt{product}\; \texttt{'*'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{product}\; \texttt{'/'} \;\texttt{factor} \\
                   & \mid        & \;\texttt{factor}                                   \\[0.2cm]
  \texttt{factor}  & \rightarrow &   \texttt{'('} \; \texttt{expr} \;\texttt{')'}      \\
                   & \mid        & \;\texttt{NUMBER}                                   \\
                   & \mid        & \;\texttt{IDENTIFIER}                               
\end{array}
$$

## Specification of the Scanner

In [ ]:
#r "CalculatorSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "CalculatorSyntax/bin/Debug/net9.0/CalculatorSyntax.dll"
open CalculatorSyntax

The FsYacc grammar declares a token case for every operator and delimiter, and payload types for numbers and identifiers.

The scanner uses named token cases rather than Python literal tokens. Its source is built by FsLex.

```fsharp
{
open CalculatorSyntax.Parser
open FSharp.Text.Lexing
let lexeme (buffer: LexBuffer<char>) = System.String(buffer.Lexeme)
}
let digit = ['0'-'9']
let natural = '0' | ['1'-'9'] digit*
let letter = ['a'-'z' 'A'-'Z']
rule token = parse
 | [' ' '\t' '\r']+ { token lexbuf }
 | '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
 | natural ('.' digit+)? { NUMBER(System.Double.Parse(lexeme lexbuf, System.Globalization.CultureInfo.InvariantCulture)) }
 | letter (letter | digit | '_')* { IDENTIFIER(lexeme lexbuf) }
 | ":=" { ASSIGN }
 | '+' { PLUS } | '-' { MINUS } | '*' { TIMES } | '/' { DIVIDE }
 | '(' { LPAREN } | ')' { RPAREN } | ';' { SEMICOLON }
 | eof { EOF }
 | _ { failwithf "Illegal character at line %d, column %d" lexbuf.StartPos.Line (lexbuf.StartPos.Column + 1) }

```

The token `NUMBER` specifies a *floating point number*.

Blanks and tabulators are ignored.

Unkown characters are reported as lexical errors.

FsLexYacc is built as a normal F# project; notebook-specific `__file__` tricks are unnecessary.

The build generates `Lexer.fs` from `Lexer.fsl`.

## Specification of the Parser

The *start variable* of our grammar is `statement`.

FsYacc generates the parser from this specification. Its semantic actions construct a typed expression tree; evaluation stays separate from parsing.

```fsharp
%{
open CalculatorSyntax
%}
%token <float> NUMBER
%token <string> IDENTIFIER
%token PLUS MINUS TIMES DIVIDE LPAREN RPAREN ASSIGN SEMICOLON EOF
%start statement
%type <CalculatorSyntax.Statement> statement
%%
statement: IDENTIFIER ASSIGN expr SEMICOLON EOF { Assign($1, $3) }
         | expr SEMICOLON EOF { Evaluate $1 }
expr: expr PLUS product { Binary(Add, $1, $3) }
    | expr MINUS product { Binary(Subtract, $1, $3) }
    | product { $1 }
product: product TIMES factor { Binary(Multiply, $1, $3) }
       | product DIVIDE factor { Binary(Divide, $1, $3) }
       | factor { $1 }
factor: LPAREN expr RPAREN { $2 }
      | NUMBER { Number $1 }
      | IDENTIFIER { Variable $1 }

```

The immutable `names2Values` map associates variable names with values. Each assignment returns an updated map; a fresh calculation session starts with an empty map.

In [ ]:
let names2Values : Map<string, float> = Map.empty

The FsYacc `statement` production implements the following grammar rule:
$$ \texttt{stmnt} \rightarrow \texttt{IDENTIFIER}\; \texttt{':='} \; \texttt{expr} \; \texttt{';'}$$

The FsYacc `statement` production implements the following grammar rule:
$$ \texttt{stmnt} \rightarrow \texttt{expr} \; \texttt{';'} $$

An *expr* is a sequence of *products* that are combined with the operator `+`.
The corresponding grammar rules are:
```
    expr : expr '+' product
         | expr '-' product
         | product
         ;
```

A *product* is a sequence of *factors* that are combined with the operator `*`.
The corresponding grammar rules are:
```
    product : product '*' factor
            | product '/' factor
            | factor
            ;
```

A factor is either an expression in parenthesis or a number.
```
   factor : '(' expr ')'
          | NUMBER
          | IDENTIFIER
          ;
```
In the case that `factor` is an `IDENTIFIER` we have to look up the value that is
associated with the corresponding variable name in the map `names2Values`.

In [ ]:
let rec evaluate names = function
    | Number value -> value
    | Variable name ->
        match Map.tryFind name names with
        | Some value -> value
        | None -> failwithf "Undefined variable: %s" name
    | Binary(op, left, right) ->
        let l, r = evaluate names left, evaluate names right
        match op with
        | Add -> l + r | Subtract -> l - r | Multiply -> l * r | Divide -> divide l r

let execute names statement =
    match statement with
    | Assign(name, expr) -> Map.add name (evaluate names expr) names, None
    | Evaluate expr -> names, Some(evaluate names expr)

let calculate names source = execute names (Parse.statement source)

The parse wrapper reports syntax errors with line and column. Unknown variables are reported by the evaluator.

`Parse.statement source` creates a typed statement. `calculate names source` then evaluates it and returns the updated map and optional result. Call `main ()` explicitly to start an interactive calculator.

In [ ]:
// Kept separate from the demonstrations so running all cells never blocks for input.
let main () =
    let rec loop names =
        printf "calc> "
        let source = System.Console.ReadLine()
        if not (System.String.IsNullOrEmpty source) then
            try
                let updated, value = calculate names source
                value |> Option.iter (printfn "%g")
                loop updated
            with error -> printfn "%s" error.Message; loop names
    loop Map.empty

In [ ]:
let assigned, _ = calculate names2Values "x := 6;"
let names, result = calculate assigned "x * 7;"
checkNotebook (result = Some 42.0)
printfn "%A" result

In [ ]:
printfn "%A" names